# Regression
- This script is used to conduct regressions.
- Simulations: CNTL, TranAlbe

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.panel import PanelOLS
home_path = '/gws/ssde/j25a/duicv/yuansun/'
project_path = f'{home_path}0_wrf-cstm_GM-HK/'
met_var_list = ['FLDS', 'FSDS', 'PBOT', 'QBOT', 'RAIN', 'TBOT', 'WIND', 'ZBOT']
region='HK'
start_date = '2035-04-01'
end_date = '2039-12-30'

In [2]:
ds_mask_region = xr.open_dataset(f'{project_path}{region}/mask/mask_{region}_lat_lon.nc')
ds_mask_region_pct_urban = ds_mask_region['PCT_URBAN']
tmax_time_zone = 6

In [3]:
for scenario in ['cntl', 'tran_albe']:
    wrfout_path = f'{project_path}runs/TranUrbAlb_{region}/runs/d04_{scenario}/archive_ctsm/lnd/'
    wrfout_path_files = os.listdir(wrfout_path)
    nc_files = [f for f in wrfout_path_files if f.endswith('.nc')  and '.clm2.h2.' in f]
    total_results = []
    for files in nc_files:
        ds = xr.open_dataset(f'{wrfout_path}{files}')
        var_max_average_list = []
        for var in met_var_list:
            ds_var = ds[var].where(ds_mask_region['mask'] == 1)
            ds_var = ds_var.assign_coords(time=ds_var.time.dt.round("h"))
            ds_var_average = (ds_var * ds_mask_region_pct_urban).sum(dim=['lat', 'lon'], skipna=True) / ds_mask_region_pct_urban.sum(dim=['lat', 'lon'], skipna=True)
            df_var_average = ds_var_average.to_dataframe(var)
            var_max_average_list.append(df_var_average)
        df_file = pd.concat(var_max_average_list, axis=1)
        total_results.append(df_file)
    df_var_max_average_total = pd.concat(total_results, axis=0).reset_index()
    df_var_max_average_total['time'] = pd.to_datetime(df_var_max_average_total['time'])     
    df_var_max_clean = df_var_max_average_total.sort_values(by='time')
    output_filename = f'./data_for_figure/{region}_met_{scenario}_urban_rural.csv'
    df_var_max_clean.to_csv(output_filename, index=False)    
    df_var_max_clean

In [2]:
# delta meterological variables
df_met_cntl = pd.read_csv(f'./data_for_figure/{region}_met_cntl_urban_rural.csv')
df_met_tran_albe = pd.read_csv(f'./data_for_figure/{region}_met_tran_albe_urban_rural.csv')
df_met_delta = df_met_tran_albe.copy()
for var in met_var_list:
    df_met_delta[var] = df_met_tran_albe[var] - df_met_cntl[var]
   
df_met_delta['time'] = pd.to_datetime(df_met_delta['time'])
df_met_delta_filtered = df_met_delta[(df_met_delta['time'] >= start_date) & (df_met_delta['time'] <= end_date)].copy() 
df_met_delta_filtered.reset_index(drop=True, inplace=True)
df_met_delta_filtered_day = df_met_delta_filtered[df_met_delta_filtered['time'].dt.hour == 6].copy() # local 14:00
df_met_delta_filtered_night = df_met_delta_filtered[df_met_delta_filtered['time'].dt.hour == 18].copy() # local 02:00
df_met_delta_filtered_day.reset_index(drop=True, inplace=True)
df_met_delta_filtered_night.reset_index(drop=True, inplace=True)
df_met_delta_filtered_day.head()

,time,FLDS,FSDS,PBOT,QBOT,RAIN,TBOT,WIND,ZBOT
0,2035-04-01 06:00:00,-72.831835,486.555719,395.999316,-0.007769,-1.723570e-05,-0.731552,-0.076106,-0.574248
1,2035-04-02 06:00:00,25.824007,153.100233,92.188385,0.001573,1.754294e-12,3.106989,2.364163,0.523950
2,2035-04-03 06:00:00,45.046894,327.781814,-432.565967,0.007103,-1.261016e-07,9.971663,-2.680560,1.849432
3,2035-04-04 06:00:00,110.614661,183.981863,-935.217011,0.010545,0.000000e+00,14.212561,-4.624648,2.585236
4,2035-04-05 06:00:00,162.910183,156.333542,-1258.078713,0.011867,0.000000e+00,18.894299,-2.673047,3.265123


In [11]:
df_met_delta_filtered_day

,time,FLDS,FSDS,PBOT,QBOT,RAIN,TBOT,WIND,ZBOT,albedo,TSA_urban,TG_urban,TSA_rural
0,2035-04-01 06:00:00,-72.635031,488.953803,395.535979,-0.007768,-2.387424e-05,-0.723471,-0.079434,-0.572581,0.524120,1.539306,11.697876,0.294525
1,2035-04-02 06:00:00,25.789944,154.908757,92.278148,0.001570,2.106630e-12,3.106328,2.315051,0.523610,0.541590,4.327881,10.028259,3.364196
2,2035-04-03 06:00:00,44.703970,326.974057,-432.167563,0.007097,-1.169493e-07,9.949001,-2.671883,1.845769,0.559061,12.155792,21.781496,10.540344
3,2035-04-04 06:00:00,110.536212,183.494629,-934.790744,0.010524,0.000000e+00,14.179829,-4.592460,2.579247,0.576532,16.353638,26.034210,14.578797
4,2035-04-05 06:00:00,162.626656,155.970555,-1257.464417,0.011807,0.000000e+00,18.861869,-2.624767,3.256552,0.594002,20.828522,29.931457,19.126709
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728,2039-12-25 06:00:00,-69.874015,-149.046824,729.017859,-0.008269,-1.557499e-06,-4.909160,-3.267584,-1.105493,15.474201,-4.934690,-5.058716,-5.127960
1729,2039-12-26 06:00:00,-88.357104,442.280247,397.670186,-0.009887,-5.943101e-04,-4.214948,4.730194,-1.089407,15.474201,-2.832153,3.224486,-3.340484
1730,2039-12-27 06:00:00,-59.045285,-171.687387,164.598783,-0.003883,-5.547345e-08,-7.919305,2.044226,-1.275769,15.474201,-8.002380,-9.190553,-7.891418
1731,2039-12-28 06:00:00,-62.021276,-117.077103,-19.505922,-0.006202,-2.202082e-06,-5.069794,1.101528,-1.008058,15.474201,-4.996947,-5.785767,-4.990723


In [3]:
# delta roof albedo (method1: using input)
df_roof_albe = pd.read_csv('../urban_fraction_albedo_scenario/data_for_figure/sum_roof_albedo_area.csv')
df_roof_albe_region = df_roof_albe[df_roof_albe['domain'] == region].copy()
df_roof_albe_region_delta = df_roof_albe_region.copy()
df_roof_albe_region_delta['sum_roof_albedo_area'] = df_roof_albe_region['sum_roof_albedo_area'] - df_roof_albe_region['sum_roof_albedo_area'].iloc[0]
df_roof_albe_region_delta['time'] = pd.to_datetime(df_roof_albe_region_delta['time'])
df_roof_albe_region_delta_filtered = df_roof_albe_region_delta[(df_roof_albe_region_delta['time'] >= start_date)].copy()
df_roof_albe_region_delta_filtered.drop(columns=['domain'], inplace=True)
df_daily = df_roof_albe_region_delta_filtered.set_index('time').resample('1D').asfreq()
df_daily = df_daily.drop(pd.Timestamp('2036-02-29'), errors='ignore')
df_daily['sum_roof_albedo_area'] = df_daily['sum_roof_albedo_area'].interpolate(method='linear')
df_daily.reset_index(inplace=True)
df_daily_filtered = df_daily[(df_daily['time'] >= start_date) & (df_daily['time'] < end_date)]
df_daily_filtered

,time,sum_roof_albedo_area
0,2035-04-01,0.743386
1,2035-04-02,0.768165
2,2035-04-03,0.792945
3,2035-04-04,0.817724
4,2035-04-05,0.842504
...,...,...
1728,2039-12-25,21.947782
1729,2039-12-26,21.947782
1730,2039-12-27,21.947782
1731,2039-12-28,21.947782


In [4]:
region='HK'
ds_mask_region = xr.open_dataset(f'{project_path}{region}/mask/mask_{region}_lat_lon.nc')
ds_mask_region_pct_urban = ds_mask_region['PCT_URBAN']
urban_mask_region = ds_mask_region_pct_urban > 0

In [6]:
tem_var_list = ['TSA', 'TG']
for scenario in ['cntl', 'tran_albe']:
    wrfout_path = f'{project_path}runs/TranUrbAlb_{region}/runs/d04_{scenario}/archive_ctsm/lnd/'
    wrfout_path_files = os.listdir(wrfout_path)
    nc_files = [f for f in wrfout_path_files if f.endswith('.nc')  and '.clm2.h0.' in f]
    total_urban_results = []
    for files in nc_files:
        ds = xr.open_dataset(f'{wrfout_path}{files}')
        ds = ds.assign_coords(time=ds['time'].dt.round("h"))
        urban_average_list = []
        for var in tem_var_list:
            ds_var_urban = ds[f'{var}_U'].where(urban_mask_region)
            ds_var_urban_average = ds_var_urban.mean(dim=['lat', 'lon'], skipna=True) - 273.15
            df_var_urban_average = ds_var_urban_average.to_dataframe(var).reset_index()
            urban_average_list.append(df_var_urban_average.rename(columns={var: f'{var}_urban'}))
        df_var_urban_average = pd.merge(urban_average_list[0], urban_average_list[1], on='time', how='inner')
        total_urban_results.append(df_var_urban_average)
    df_urban_all = pd.concat(total_urban_results, axis=0, ignore_index=True)
    output_filename = f'./data_for_figure/{region}_temp_{scenario}_urban.csv'
    df_urban_all['time'] = pd.to_datetime(df_urban_all['time'])
    df_urban_clean = df_urban_all.sort_values('time').reset_index(drop=True)
    df_urban_clean.to_csv(output_filename, index=False)
    df_urban_clean

In [7]:
df_temp_cntl = pd.read_csv(f'./data_for_figure/HK_temp_cntl_urban.csv')
df_temp_tran_albe = pd.read_csv(f'./data_for_figure/HK_temp_tran_albe_urban.csv')
df_temp_delta = df_temp_tran_albe[['time', 'TSA_urban', 'TG_urban']].copy()
df_temp_delta['TSA_urban'] = df_temp_tran_albe['TSA_urban'] - df_temp_cntl['TSA_urban']
df_temp_delta['TG_urban'] = df_temp_tran_albe['TG_urban'] - df_temp_cntl['TG_urban']
df_temp_delta['time'] = pd.to_datetime(df_temp_delta['time'])
df_temp_delta_filtered_day = df_temp_delta[(df_temp_delta['time'] >= start_date) & (df_temp_delta['time'] < end_date) & (df_temp_delta['time'].dt.hour == 6)] 
df_temp_delta_filtered_day.reset_index(drop=True, inplace=True)
df_temp_delta_filtered_night = df_temp_delta[(df_temp_delta['time'] >= start_date) & (df_temp_delta['time'] < end_date) & (df_temp_delta['time'].dt.hour == 18)]
df_temp_delta_filtered_night.reset_index(drop=True, inplace=True)
df_temp_delta_filtered_day

,time,TSA_urban,TG_urban
0,2035-04-01 06:00:00,1.533356,11.654480
1,2035-04-02 06:00:00,4.318635,9.970520
2,2035-04-03 06:00:00,12.182160,21.824584
3,2035-04-04 06:00:00,16.394562,26.096832
4,2035-04-05 06:00:00,20.878448,30.027130
...,...,...,...
1728,2039-12-25 06:00:00,-4.854492,-4.485382
1729,2039-12-26 06:00:00,-2.801422,3.501372
1730,2039-12-27 06:00:00,-7.951478,-8.772003
1731,2039-12-28 06:00:00,-4.913209,-5.081787


In [8]:
df_met_delta_filtered_day['albedo'] = df_daily_filtered['sum_roof_albedo_area'].values
df_met_delta_filtered_day['TSA_urban'] = df_temp_delta_filtered_day['TSA_urban'].values
df_met_delta_filtered_day['TG_urban'] = df_temp_delta_filtered_day['TG_urban'].values
df_met_delta_filtered_day_period = df_met_delta_filtered_day[(df_met_delta_filtered_day['time'].dt.month >=4) & (df_met_delta_filtered_day['time'].dt.month <=9)].copy()
df_met_delta_filtered_night['albedo'] = df_daily_filtered['sum_roof_albedo_area'].values
df_met_delta_filtered_night['TSA_urban'] = df_temp_delta_filtered_night['TSA_urban'].values
df_met_delta_filtered_night['TG_urban'] = df_temp_delta_filtered_night['TG_urban'].values
df_met_delta_filtered_night_period = df_met_delta_filtered_night[(df_met_delta_filtered_night['time'].dt.month >=4) & (df_met_delta_filtered_night['time'].dt.month <=9)].copy()
df_met_delta_filtered_day_period

,time,FLDS,FSDS,PBOT,QBOT,RAIN,TBOT,WIND,ZBOT,albedo,TSA_urban,TG_urban
0,2035-04-01 06:00:00,-72.831835,486.555719,395.999316,-0.007769,-1.723570e-05,-0.731552,-0.076106,-0.574248,0.743386,1.533356,11.654480
1,2035-04-02 06:00:00,25.824007,153.100233,92.188385,0.001573,1.754294e-12,3.106989,2.364163,0.523950,0.768165,4.318635,9.970520
2,2035-04-03 06:00:00,45.046894,327.781814,-432.565967,0.007103,-1.261016e-07,9.971663,-2.680560,1.849432,0.792945,12.182160,21.824584
3,2035-04-04 06:00:00,110.614661,183.981863,-935.217011,0.010545,0.000000e+00,14.212561,-4.624648,2.585236,0.817724,16.394562,26.096832
4,2035-04-05 06:00:00,162.910183,156.333542,-1258.078713,0.011867,0.000000e+00,18.894299,-2.673047,3.265123,0.842504,20.878448,30.027130
...,...,...,...,...,...,...,...,...,...,...,...,...
1638,2039-09-26 06:00:00,-32.334304,-50.999850,1562.708468,-0.005697,-4.119915e-07,-1.860669,-2.164298,-0.550665,21.947782,-3.274597,-10.032136
1639,2039-09-27 06:00:00,-39.392118,71.421136,1601.700503,-0.006486,-8.596556e-06,-1.611531,-3.100351,-0.518406,21.947782,-2.448608,-6.072265
1640,2039-09-28 06:00:00,-32.567829,-19.316503,1702.605147,-0.005214,-1.997501e-13,-1.345097,-1.792281,-0.401331,21.947782,-2.582184,-8.445435
1641,2039-09-29 06:00:00,-18.580214,-5.453631,1620.171817,-0.003885,8.825560e-05,-0.081292,-1.655021,-0.131643,21.947782,-0.675840,-3.318940


In [9]:
# at 14:00
df_reg = df_met_delta_filtered_day_period.copy()
X = df_reg[['FSDS','FLDS','PBOT','QBOT','RAIN','TBOT','WIND','ZBOT','albedo']] #, 'TG_urban'
vif = pd.DataFrame() # variance inflation factor (VIF)
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(np.round(vif,2))

  variable      VIF
0     FSDS     3.30
1     FLDS    28.92
2     PBOT     7.48
3     QBOT   255.20
4     RAIN     1.36
5     TBOT  1266.86
6     WIND     1.42
7     ZBOT  2090.82
8   albedo     1.75


In [30]:
df_reg = df_met_delta_filtered_day_period.copy()
X = df_reg[['FSDS','FLDS','RAIN', 'WIND', 'albedo']] # remove: 'QBOT', 'TBOT', 'ZBOT', 'PBOT',
vif = pd.DataFrame() # variance inflation factor (VIF)
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(np.round(vif,2))
# 1: no multicollinearity issue
# 1-5: moderate multicollinearity
# >5: high multicollinearity

  variable   VIF
0     FSDS  1.24
1     FLDS  1.72
2     RAIN  1.20
3     WIND  1.08
4   albedo  1.70


In [31]:
df_reg = df_met_delta_filtered_night_period.copy()
X = df_reg[['FLDS','PBOT','QBOT','RAIN','TBOT','WIND','ZBOT','albedo']] # , 'TG_urban'
vif = pd.DataFrame() # variance inflation factor (VIF)
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(np.round(vif,2))

  variable      VIF
0     FLDS    25.39
1     PBOT     7.81
2     QBOT   277.37
3     RAIN     1.12
4     TBOT  1371.81
5     WIND     1.61
6     ZBOT  2310.16
7   albedo     1.70


In [32]:
df_reg = df_met_delta_filtered_night_period.copy()
X = df_reg[['FLDS','RAIN', 'WIND', 'albedo']] # remove: 'QBOT', 'TBOT', 'ZBOT', 'PBOT',
vif = pd.DataFrame() # variance inflation factor (VIF)
vif["variable"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(np.round(vif,2))

  variable   VIF
0     FLDS  1.63
1     RAIN  1.04
2     WIND  1.13
3   albedo  1.59


# Regression

In [ ]:
# annual
scaler = StandardScaler()
y = df_met_delta_filtered_day_period['TG_urban']
df_reg = df_met_delta_filtered_day_period.copy()
X = df_reg[['FSDS','FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'PBOT',
X_scaled = scaler.fit_transform(X) # standardized before regression
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               TG_urban   R-squared:                       0.818
Model:                            OLS   Adj. R-squared:                  0.817
Method:                 Least Squares   F-statistic:                     816.2
Date:                Sun, 04 Jan 2026   Prob (F-statistic):               0.00
Time:                        06:42:09   Log-Likelihood:                -2472.6
No. Observations:                 915   AIC:                             4957.
Df Residuals:                     909   BIC:                             4986.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.7152      0.120     47.746      0.0

In [34]:
scaler = StandardScaler()
y = df_met_delta_filtered_night_period['TG_urban']
df_reg = df_met_delta_filtered_night_period.copy()
X = df_reg[['FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'FSDS', 'TBOT','PBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               TG_urban   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.864
Method:                 Least Squares   F-statistic:                     1454.
Date:                Sun, 04 Jan 2026   Prob (F-statistic):               0.00
Time:                        06:43:16   Log-Likelihood:                -1988.1
No. Observations:                 915   AIC:                             3986.
Df Residuals:                     910   BIC:                             4010.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.1826      0.070    101.960      0.0

In [35]:
scaler = StandardScaler()
y = df_met_delta_filtered_day_period['TSA_urban']
df_reg = df_met_delta_filtered_day_period.copy()
X = df_reg[['FSDS','FLDS','RAIN','WIND', 'albedo']] # 'QBOT', 'TBOT', 'PBOT', 'ZBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              TSA_urban   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.840
Method:                 Least Squares   F-statistic:                     958.0
Date:                Sun, 04 Jan 2026   Prob (F-statistic):               0.00
Time:                        06:44:39   Log-Likelihood:                -2003.4
No. Observations:                 915   AIC:                             4019.
Df Residuals:                     909   BIC:                             4048.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.8392      0.072     81.465      0.0

In [36]:
scaler = StandardScaler()
y = df_met_delta_filtered_night_period['TSA_urban']
df_reg = df_met_delta_filtered_night_period.copy()
X = df_reg[['FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'FSDS','TBOT','PBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              TSA_urban   R-squared:                       0.832
Model:                            OLS   Adj. R-squared:                  0.831
Method:                 Least Squares   F-statistic:                     1124.
Date:                Sun, 04 Jan 2026   Prob (F-statistic):               0.00
Time:                        06:45:44   Log-Likelihood:                -2069.3
No. Observations:                 915   AIC:                             4149.
Df Residuals:                     910   BIC:                             4173.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.9379      0.077     90.116      0.0

# Backup

In [22]:
df_met_delta_filtered_day_jja = df_met_delta_filtered_day[(df_met_delta_filtered_day['time'].dt.month >=6) & (df_met_delta_filtered_day['time'].dt.month <=8)].copy()
df_met_delta_filtered_night_jja = df_met_delta_filtered_night[(df_met_delta_filtered_night['time'].dt.month >=6) & (df_met_delta_filtered_night['time'].dt.month <=8)].copy()

In [ ]:
# jja
scaler = StandardScaler()
y = df_met_delta_filtered_day_jja['TG_urban']
df_reg = df_met_delta_filtered_day_jja.copy()
X = df_reg[['FSDS','FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'PBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               TG_urban   R-squared:                       0.779
Model:                            OLS   Adj. R-squared:                  0.776
Method:                 Least Squares   F-statistic:                     319.9
Date:                Tue, 30 Dec 2025   Prob (F-statistic):          2.91e-146
Time:                        04:52:56   Log-Likelihood:                -1176.4
No. Observations:                 460   AIC:                             2365.
Df Residuals:                     454   BIC:                             2390.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.6955      0.147     38.874      0.0

In [25]:
# jja
scaler = StandardScaler()
y = df_met_delta_filtered_night_jja['TG_urban']
df_reg = df_met_delta_filtered_night_jja.copy()
X = df_reg[['FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'PBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               TG_urban   R-squared:                       0.845
Model:                            OLS   Adj. R-squared:                  0.844
Method:                 Least Squares   F-statistic:                     621.9
Date:                Tue, 30 Dec 2025   Prob (F-statistic):          7.02e-183
Time:                        04:54:18   Log-Likelihood:                -876.63
No. Observations:                 460   AIC:                             1763.
Df Residuals:                     455   BIC:                             1784.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.9753      0.076     91.446      0.0

In [26]:
# jja
scaler = StandardScaler()
y = df_met_delta_filtered_day_jja['TSA_urban']
df_reg = df_met_delta_filtered_day_jja.copy()
X = df_reg[['FSDS','FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'PBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              TSA_urban   R-squared:                       0.812
Model:                            OLS   Adj. R-squared:                  0.809
Method:                 Least Squares   F-statistic:                     391.0
Date:                Tue, 30 Dec 2025   Prob (F-statistic):          5.79e-162
Time:                        04:54:55   Log-Likelihood:                -896.75
No. Observations:                 460   AIC:                             1805.
Df Residuals:                     454   BIC:                             1830.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.8411      0.080     73.219      0.0

In [27]:
# jja
scaler = StandardScaler()
y = df_met_delta_filtered_night_jja['TSA_urban']
df_reg = df_met_delta_filtered_night_jja.copy()
X = df_reg[['FLDS','RAIN','WIND','albedo']] # 'QBOT', 'ZBOT', 'PBOT',
X_scaled = scaler.fit_transform(X)
X_scaled = sm.add_constant(X_scaled)  # add intercept
model = sm.OLS(y, X_scaled).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              TSA_urban   R-squared:                       0.800
Model:                            OLS   Adj. R-squared:                  0.798
Method:                 Least Squares   F-statistic:                     454.2
Date:                Tue, 30 Dec 2025   Prob (F-statistic):          2.45e-157
Time:                        04:55:37   Log-Likelihood:                -931.23
No. Observations:                 460   AIC:                             1872.
Df Residuals:                     455   BIC:                             1893.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.6725      0.086     77.685      0.0